In [ ]:
!pip install -q dbt-duckdb

In [ ]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_name = vl.lakehouse_name
    dbt_path       = vl.dbt_path
    metadata_path  = vl.metadata_path
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
except ModuleNotFoundError:
    import yaml
    from pathlib import Path
    _root  = Path("C:/lakehouse/default/Files/dbt_fabric_python_notebook")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws_id"]
    lakehouse_id   = _cfg["lakehouse_id"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    metadata_path  = _cfg["metadata_path"]
os.environ['ROOT_PATH']      = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}'
os.environ['download_limit'] = download_limit
os.environ['process_limit']  = process_limit

In [ ]:
from datetime import datetime, timedelta, timezone
from dbt.cli.main import dbtRunner

LOCAL_METADATA    = "/tmp/metadata.db"
STALE_LEASE_HOURS = 12

def _run_dbt():
    os.chdir(dbt_path)
    dbt = dbtRunner()
    dbt.invoke(["run",  "--target", "prod", "--profiles-dir", "."])
    dbt.invoke(["test", "--target", "prod", "--profiles-dir", "."])

try:
    import notebookutils  # noqa: F401
    _is_fabric = True
except ModuleNotFoundError:
    _is_fabric = False

if not _is_fabric:
    os.environ['METADATA_LOCAL_PATH'] = metadata_path
    _run_dbt()
else:
    from azure.storage.filedatalake import DataLakeServiceClient, DataLakeLeaseClient
    from azure.core.credentials import AccessToken, TokenCredential
    from azure.core.exceptions import ResourceNotFoundError, HttpResponseError

    class _StaticToken(TokenCredential):
        def __init__(self, t): self.t = t
        def get_token(self, *_, **__): return AccessToken(self.t, 9999999999)

    marker = "/Files/"
    idx    = metadata_path.find(marker)
    if idx < 0:
        raise ValueError(f"metadata_path missing '/Files/' segment: {metadata_path}")
    rel = f"{lakehouse_id}/Files/{metadata_path[idx + len(marker):]}"

    file = (
        DataLakeServiceClient(
            "https://onelake.dfs.fabric.microsoft.com",
            credential=_StaticToken(notebookutils.credentials.getToken("storage")),
        )
        .get_file_system_client(workspace_id)
        .get_file_client(rel)
    )

    try:
        file.get_file_properties()
    except ResourceNotFoundError:
        file.upload_data(b"", overwrite=True)

    try:
        lease = file.acquire_lease(lease_duration=-1)
    except HttpResponseError as e:
        if e.error_code != "LeaseAlreadyPresent":
            raise
        props      = file.get_file_properties()
        stamped    = (props.metadata or {}).get("acquired_at")
        is_stale   = True
        if stamped:
            try:
                age = datetime.now(timezone.utc) - datetime.fromisoformat(stamped)
                is_stale = age > timedelta(hours=STALE_LEASE_HOURS)
            except ValueError:
                is_stale = True
        if not is_stale:
            raise RuntimeError(
                f"metadata.db is locked by an active writer (acquired_at={stamped}). "
                f"Refusing to run. Stale leases auto-break after {STALE_LEASE_HOURS}h."
            )
        DataLakeLeaseClient(file).break_lease(lease_break_period=0)
        lease = file.acquire_lease(lease_duration=-1)

    try:
        file.set_metadata(
            {"acquired_at": datetime.now(timezone.utc).isoformat(), "holder": "fabric-notebook"},
            lease=lease.id,
        )
        with open(LOCAL_METADATA, "wb") as fh:
            fh.write(file.download_file(lease=lease.id).readall())
        os.environ['METADATA_LOCAL_PATH'] = LOCAL_METADATA
        try:
            _run_dbt()
        finally:
            with open(LOCAL_METADATA, "rb") as fh:
                file.upload_data(fh.read(), overwrite=True, lease=lease.id)
    finally:
        lease.release()